# 📓 Notebook 24 — DeepAgents Library---**Phase 8 · Deep Agent Patterns**> **DeepAgents** is LangChain's production agent harness — built on top of LangChain + LangGraph — that transforms "shallow" agents into **deep agents** capable of handling complex, multi-step tasks that ordinary agents fail at.### Shallow Agents vs Deep Agents| | Shallow Agent | Deep Agent (DeepAgents) ||--|--------------|----------------------|| Planning | None / one-shot | `write_todos` — dynamic task tracking || Context | Grows until overflow | Filesystem backend — offload to storage || Delegation | None | Subagent spawning with context isolation || Memory | Chat history only | Long-term memory across sessions || Auditability | Opaque decisions | Full trace via LangSmith || Complex tasks | Fails after 5-10 steps | Handles 50+ step workflows |### What You'll Learn| Topic | Why It Matters ||-------|---------------|| `create_deep_agent` | The main API to build a deep agent || Todo middleware | Agent plans, tracks, and adapts its own work || Filesystem middleware | Offload context to storage, avoid overflow || Subagent middleware | Spawn specialized agents for subtasks || Summarization | Compress conversation to stay within limits || Custom tools | Extend deep agents with your own tools || LangGraph integration | Checkpointing, streaming, human-in-the-loop |### 🏗️ Build: Deep Agent for Complex Research Tasks> **Install:** `pip install deepagents`

## 1. Setup

In [ ]:
# pip install deepagentsimport osfrom dotenv import load_dotenvload_dotenv()# Verify API keyapi_key_type = Noneif os.getenv("OPENAI_API_KEY"):    api_key_type = "OpenAI"elif os.getenv("ANTHROPIC_API_KEY"):    api_key_type = "Anthropic"elif os.getenv("GOOGLE_API_KEY"):    api_key_type = "Google"print(f"🔑 Using: {api_key_type or 'No API key found — set one in .env'}")# Enable LangSmith tracing (recommended)os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")os.environ.setdefault("LANGCHAIN_PROJECT", "deepagents-course")

---## 2. Your First Deep Agent### `create_deep_agent` — The Core API```pythonfrom deepagents import create_deep_agentagent = create_deep_agent(    model=model,             # Any LangChain chat model    tools=[...],             # Custom tools (optional)    system_prompt="...",     # Agent's role & instructions    subagents=[...],         # Specialist sub-agents (optional))```The deep agent comes with **built-in middleware**:1. **TodoListMiddleware** — The agent can plan by writing todos2. **FilesystemMiddleware** — Read/write files to manage large context3. **SubAgentMiddleware** — Spawn subagents for focused subtasks4. **SummarizationMiddleware** — Compress conversation history

In [ ]:
from deepagents import create_deep_agent# Choose your LLMif os.getenv("OPENAI_API_KEY"):    from langchain_openai import ChatOpenAI    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)elif os.getenv("ANTHROPIC_API_KEY"):    from langchain_anthropic import ChatAnthropic    model = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)elif os.getenv("GOOGLE_API_KEY"):    from langchain_google_genai import ChatGoogleGenerativeAI    model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)# Create a basic deep agentagent = create_deep_agent(    model=model,    system_prompt="You are a helpful research assistant. Break complex tasks into steps, track progress with todos, and use the filesystem to manage large outputs.")print(f"✅ Deep Agent created with model: {model.__class__.__name__}")print(f"   Built-in tools: write_todos, read_file, write_file, ls, edit_file, glob, grep")

---## 3. The Todo System — Planning That Adapts### How `write_todos` WorksThe deep agent has a built-in `write_todos` tool that lets it:1. **Create a plan** as a todo list at the start of a task2. **Check off items** as they complete3. **Modify the plan** when new information emerges4. **Track progress** — know what's done vs remaining```Agent receives complex task    ↓Uses write_todos to create a plan:  ☐ Step 1 — Research the topic  ☐ Step 2 — Analyze the data  ☐ Step 3 — Write the report    ↓Executes Step 1 → marks as done:  ☑ Step 1 — Research the topic  ☐ Step 2 — Analyze the data  ☐ Step 3 — Write the report    ↓Discovers new info → adapts plan:  ☑ Step 1 — Research the topic  ☐ Step 2a — Clean the data first  ☐ Step 2b — Analyze the cleaned data  ☐ Step 3 — Write the report```> **Interview Tip:** This is what makes deep agents "deep" — they don't just react, they **plan, track, and adapt**. A shallow agent would try to do everything in one LLM call and lose track.

In [ ]:
# Demo: Deep agent with planningfrom langchain_core.messages import HumanMessage# Invoke with a complex taskresult = agent.invoke({    "messages": [HumanMessage(content="""Research and compare the top 3 Python web frameworks (Django, Flask, FastAPI) across these dimensions:1. Performance benchmarks2. Learning curve3. Best use cases4. Production adoptionWrite a structured comparison report.""")]})# The agent will:# 1. Use write_todos to create a plan# 2. Research each framework systematically# 3. Write findings to files to manage context# 4. Synthesize into a final reportprint("📝 Agent Response:")print(result["messages"][-1].content[:500])print("...")

---## 4. Filesystem Middleware — Context Management### The Context Overflow ProblemShallow agents stuff everything into the conversation history:- 10 tool results × 2000 tokens each = 20,000 tokens of context- LLM gets confused, loses focus, forgets the task### Deep Agent Solution: Virtual Filesystem```Deep Agent    ↓ write_file("research/django.md", findings)    ↓ write_file("research/flask.md", findings)     ↓ write_file("research/fastapi.md", findings)    ↓ read_file("research/django.md")  ← only loads what's needed    ↓Context stays clean, agent stays focused```Built-in filesystem tools:| Tool | Description ||------|-------------|| `write_file` | Save content to virtual filesystem || `read_file` | Load content from virtual filesystem || `edit_file` | Modify parts of existing files || `ls` | List files in a directory || `glob` | Find files by pattern || `grep` | Search for text across files |> **Interview Tip:** The filesystem is a form of **external working memory**. Instead of cramming everything into the LLM's context window, the agent offloads information and retrieves it on demand.

In [ ]:
# The agent uses filesystem tools automatically# Here's what happens under the hood:# The agent might do:# 1. write_file("analysis/framework_comparison.md", "# Framework Comparison\n...")# 2. write_file("analysis/performance_data.json", '{"django": {...}, ...}')# 3. Later: read_file("analysis/performance_data.json") to reference data# You can configure the filesystem backend:from deepagents import create_deep_agent# In-memory filesystem (default — ephemeral)agent_memory = create_deep_agent(    model=model,    system_prompt="You are a research agent that organizes work into files.",    # Default backend is in-memory)print("✅ Agent with in-memory filesystem")print("   Files persist during the session but are lost after")

---## 5. Subagent Spawning — Delegation### Why Subagents?| Problem | Solution ||---------|----------|| Task too complex for one agent | Spawn specialist subagents || Context getting cluttered | Subagent has its own isolated context || Need deep dive on a subtask | Subagent focuses exclusively on it || Risk of distraction | Main agent stays on track |```Main Agent (orchestrator)    ├── Subagent 1: "Research Django performance"    │     └── Returns: summary of findings    ├── Subagent 2: "Research Flask performance"      │     └── Returns: summary of findings    └── Subagent 3: "Research FastAPI performance"          └── Returns: summary of findings    ↓Main Agent synthesizes all results```

In [ ]:
# Creating a deep agent with subagentsfrom deepagents import create_deep_agent# Define specialist subagentssubagents = [    {        "name": "code_analyzer",        "description": "Analyzes code quality, patterns, and suggests improvements",        "system_prompt": "You are an expert code reviewer. Analyze code thoroughly and provide actionable feedback.",    },    {        "name": "doc_writer",        "description": "Writes clear technical documentation from code and specifications",        "system_prompt": "You are a technical writer. Create clear, well-structured documentation.",    },    {        "name": "test_designer",        "description": "Designs test cases and testing strategies",        "system_prompt": "You are a QA engineer. Design comprehensive test cases covering edge cases.",    },]# Main agent with delegation capabilityorchestrator = create_deep_agent(    model=model,    system_prompt="""You are a senior engineering lead. For complex tasks:1. Break the work into subtasks using write_todos2. Delegate specialized work to subagents3. Synthesize results into a cohesive deliverable""",    subagents=subagents,)print("✅ Orchestrator agent with 3 subagents:")for sa in subagents:    print(f"   • {sa['name']}: {sa['description'][:50]}...")

In [ ]:
# Test subagent delegationresult = orchestrator.invoke({    "messages": [HumanMessage(content="""Review this Python function, write documentation for it, and design test cases:def fibonacci(n):    if n <= 0:        return []    elif n == 1:        return [0]    elif n == 2:        return [0, 1]    fib = [0, 1]    for i in range(2, n):        fib.append(fib[i-1] + fib[i-2])    return fib""")]})# The orchestrator will:# 1. Use write_todos to plan the work# 2. Spawn code_analyzer to review the function# 3. Spawn doc_writer to write documentation# 4. Spawn test_designer to create test cases# 5. Synthesize all resultsprint("📝 Orchestrator Result:")print(result["messages"][-1].content[:600])print("...")

---## 6. Adding Custom ToolsDeep agents can be extended with your own tools alongside the built-in ones.

In [ ]:
from langchain_core.tools import toolimport json@tooldef search_docs(query: str) -> str:    """Search technical documentation for information about Python frameworks."""    # Simulated search results    docs = {        "django": "Django is a high-level Python web framework. It includes ORM, admin panel, auth. Used by Instagram, Pinterest.",        "flask": "Flask is a lightweight WSGI framework. Minimal core, extensive extensions. Used by LinkedIn, Netflix.",        "fastapi": "FastAPI is a modern async framework. Auto OpenAPI docs, type hints, very fast. Used by Microsoft, Netflix.",        "langchain": "LangChain is a framework for building LLM applications with chains, agents, and RAG.",    }    results = []    for key, info in docs.items():        if key in query.lower():            results.append(info)    return "\n".join(results) if results else f"No docs found for: {query}"@tooldef calculate_metric(expression: str) -> str:    """Calculate a numeric expression for performance comparisons."""    try:        result = eval(expression, {"__builtins__": {}})        return f"Result: {result}"    except Exception as e:        return f"Error: {e}"# Deep agent with custom tools + built-in toolscustom_agent = create_deep_agent(    model=model,    tools=[search_docs, calculate_metric],  # Your tools    system_prompt="You are a research analyst. Use search_docs for information and calculate_metric for comparisons. Use the filesystem to organize your research.",)# The agent now has: write_todos, read_file, write_file, ls, edit_file, # glob, grep, search_docs, calculate_metricprint("✅ Custom agent with built-in + custom tools")

---## 7. LangGraph Integration — Production FeaturesDeepAgents is built on LangGraph, so you get all LangGraph features:

In [ ]:
from langgraph.checkpoint.memory import MemorySaver# Create agent with checkpointing for multi-turn conversationscheckpointer = MemorySaver()persistent_agent = create_deep_agent(    model=model,    system_prompt="You are a persistent research assistant. Remember context across conversations.",    checkpointer=checkpointer,)# Multi-turn conversation with state persistenceconfig = {"configurable": {"thread_id": "research_session_1"}}turns = [    "I'm researching the best Python web framework for a fintech API. We need high performance and type safety.",    "Based on that, what would you recommend and why?",    "Write a project plan for migrating our existing Flask app to your recommendation.",]print("💾 Multi-Turn Deep Agent (with checkpointing)")print("=" * 60)for turn in turns:    result = persistent_agent.invoke(        {"messages": [HumanMessage(content=turn)]},        config=config,    )    print(f"\n👤 {turn[:60]}...")    print(f"🤖 {result['messages'][-1].content[:150]}...")

In [ ]:
# Streaming — see tokens as they generateprint("📡 Streaming Deep Agent Response:")print("-" * 40)for event in persistent_agent.stream(    {"messages": [HumanMessage(content="Summarize your recommendations in 3 bullet points.")]},    config={"configurable": {"thread_id": "research_session_1"}},    stream_mode="messages",):    msg, metadata = event    if msg.content:        print(msg.content, end="", flush=True)print()

---## 8. Deep Agents vs Other Approaches### When to Use DeepAgents| Scenario | Use DeepAgents? | Alternative ||----------|----------------|-------------|| Complex multi-step research | ✅ Yes | — || Simple Q&A chatbot | ❌ No | Basic LangChain chain || Quick tool-calling agent | ❌ No | `create_react_agent` || Long-running workflows (50+ steps) | ✅ Yes | — || Code generation + review | ✅ Yes | — || Simple RAG pipeline | ❌ No | LangChain RAG chain || Multi-agent orchestration | ✅ Yes (subagents) | CrewAI, LangGraph |### Architecture Comparison```Shallow Agent (LangChain AgentExecutor):  User → LLM → (Tool → LLM → Tool → LLM ...) → Answer  ⚠️ Context grows linearly, no planning, no delegationDeep Agent (DeepAgents):  User → LLM → write_todos (plan)             → Tool → write_file (save results)             → spawn_subagent (delegate)             → read_file (retrieve)               → edit_todos (adapt plan)             → Answer  ✅ Managed context, planning, delegation, auditable```

---## 📝 Interview Quiz — DeepAgents### Q1: What makes a "deep" agent different from a "shallow" agent?<details><summary>💡 Answer</summary>**Shallow agent:** Receives a task → calls tools in a loop → returns answer. No planning, no context management, fails on complex tasks.**Deep agent (DeepAgents):** Has four key capabilities:1. **Planning** — `write_todos` to decompose tasks and track progress2. **Context management** — Filesystem backend to offload data, avoiding context overflow3. **Delegation** — Subagent spawning for specialized subtasks with context isolation4. **Memory** — Long-term memory across sessions via checkpointingThe result: deep agents can handle 50+ step workflows that shallow agents would lose track of.</details>### Q2: How does the filesystem middleware prevent context overflow?<details><summary>💡 Answer</summary>Instead of keeping all tool results in the conversation history (which grows and confuses the LLM), the agent:1. **Writes** results to the virtual filesystem: `write_file("research/findings.md", ...)`2. **References** only the filename in conversation: "Saved findings to research/findings.md"3. **Reads** later when needed: `read_file("research/findings.md")`This is **external working memory** — the conversation stays small and focused while all data is accessible on demand.**Backends:** In-memory (default), local disk, or custom (e.g., S3, database).</details>### Q3: When would you use subagent spawning vs just calling tools?<details><summary>💡 Answer</summary>**Use tools when:**- The task is simple (search, calculate, fetch data)- Result is small and fits in conversation- No deep reasoning needed**Use subagents when:**- The subtask requires **multi-step reasoning** (e.g., "analyze this codebase")- You need **context isolation** (subagent's work doesn't clutter main agent)- The subtask requires **different expertise** (code review vs. documentation)- You want **parallel execution** of independent subtasks**Think of it as:** Tools are like functions. Subagents are like team members.</details>### Q4: How does DeepAgents relate to LangChain and LangGraph?<details><summary>💡 Answer</summary>```┌──────────────────────────┐│       DeepAgents         │  ← Agent harness (planning, filesystem, subagents)├──────────────────────────┤│       LangGraph          │  ← Runtime (state machines, checkpointing, streaming)├──────────────────────────┤│       LangChain          │  ← Building blocks (models, tools, prompts)└──────────────────────────┘```- **LangChain** provides the primitives (chat models, tools, prompts)- **LangGraph** provides the runtime (stateful graphs, persistence, streaming)- **DeepAgents** provides the **agent-level harness** (planning, context management, delegation)DeepAgents is NOT a replacement for LangChain/LangGraph — it's built on top of them, adding the "deep" capabilities.</details>### Q5: Compare DeepAgents, CrewAI, and AutoGen for multi-agent tasks.<details><summary>💡 Answer</summary>| Aspect | DeepAgents | CrewAI | AutoGen ||--------|-----------|--------|---------|| **Model** | Main agent + subagents | Role-based team | Conversational agents || **Planning** | Built-in (write_todos) | Task assignments | Conversation flow || **Context** | Filesystem backend | Shared task context | Chat history || **Delegation** | Explicit subagent spawn | Agent delegation | Group chat || **Runtime** | LangGraph | Own runtime | Own runtime || **Complexity** | Medium | Low | Medium || **Best for** | Long, complex single-user tasks | Quick multi-agent setup | Research, code gen |**Choose DeepAgents when:** Single-user complex workflow needing planning + context management.**Choose CrewAI when:** Clear team roles, quick prototype.**Choose AutoGen when:** Multi-model conversations, research exploration.</details>

---## ✅ Notebook 24 Summary| Concept | Key Takeaway ||---------|-------------|| `create_deep_agent` | Core API — creates an agent with built-in planning, filesystem, subagents || Todo middleware | `write_todos` — agent plans, tracks, and adapts its work || Filesystem middleware | Offload context to virtual files, avoid overflow || Subagent middleware | Spawn specialists for context-isolated subtasks || Summarization | Compress conversation to stay within token limits || Custom tools | Add your own tools alongside built-in ones || LangGraph runtime | Checkpointing, streaming, human-in-the-loop for free || Deep vs shallow | Planning + context + delegation = handles complex tasks |### 🏁 Now try the [Capstone Projects](./capstone/) using DeepAgents!